# MeowMentum drop tests — attitude and angular velocity at impact

Hardware falling-cat drops, 10 Aug 2025. **18 drops**: 3 release roll angles
(45°, 90°, 180°) x 2 morphologies (**NT** = no tail, **WT** = with tail) x 3 trials.
Logs live in `NT_0810/` and `WT_0810/` next to this notebook and are written by
`hardware/controller.py` (front + back IMU quaternions, joint encoders, commands).

Three figures, all evaluated at a single instant — the moment the robot *would* hit the
floor if dropped from `DROP_HEIGHT_M`:

1. phase portrait: signed roll vs. signed roll rate at impact, one dot per drop;
2. grouped bars: body tilt at impact, no-tail vs. with-tail, by release angle;
3. grouped bars: total angular speed at impact, same grouping.

## Caveats a reader has to know

- **The back IMU is dead in all 18 files.** `B_Q0..B_Q3` are exactly `(1, 0, 0, 0)` and
  `B_ACC` is exactly `0` on every row of every file (verified). Every orientation number
  here is therefore the **front body only** — this is not whole-robot attitude. The
  `B_M2` rear-roll encoder does work, and `B_M1` (tail) moves on WT and is 0 on NT by design.
- **t = 0 is drop *detection*, not release.** Logging starts when `F_ACC < 3.5`, so t0
  carries up to one 20 ms control loop of latency, one-sided (the log starts *late*, so
  the true impact instant is up to 20 ms *earlier* in log time). Re-evaluating every drop
  at `t_impact ± 20 ms` moves the angle by 3.9° on average (15.1° worst case) and the roll
  rate by 75 °/s on average (277 °/s worst case). That is small against the trial-to-trial
  scatter for the group means, but a single dot in figure 1 can visibly move.
- **The controller drives for a fixed 1.0 s regardless of the real drop height**, so the
  robot keeps actuating past the moment it would have landed. Samples after `t_impact` are
  post-impact fiction. This is exactly why `DROP_HEIGHT_M` is a knob and not a constant:
  the NT-vs-WT ordering genuinely inverts around h ≈ 2.5 m.
- **n = 3 trials per cell.** The bars in figures 2 and 3 are means of three drops, and the
  standard deviation frequently exceeds the mean. Individual trials are drawn on top of
  every bar; read the dots, not just the bar.
- **`F_ACC` is not used to find impact.** On a spinning, actuating body it reads 1–111 m/s²
  of centripetal/tangential/vibration signal, so impact time comes from the free-fall model
  `t_impact = sqrt(2h/g)`, not from the accelerometer.
- Logs span only ~0.98 s of flight, so **heights above ~4.74 m cannot be evaluated at all**;
  drops whose log ends before `t_impact` are dropped from every figure with a printed banner,
  never extrapolated.

## How each number is computed, and where the three derivations disagreed

**Pipeline per drop:** front quaternion `(w,x,y,z)` → `scipy` rotation → de-duplicate
stale frames → unwrap → sample at `t_impact = sqrt(2h/g)`.

| step | choice | why |
|---|---|---|
| stale frames | **drop any row whose quaternion is bit-identical to the previous row's**, then work on the survivors | a stale row is the *previous* measurement stamped with a *new* time; leaving it in drags the interpolant backwards and makes a naive difference read 0 °/s then double |
| unwrapping | unwrap roll **before** any differentiation or interpolation; wrap to (−180°, 180°] only for display | 4 of 18 files cross the ±180° branch cut; a naive difference across it reads **+17 800 °/s** where the truth is −197 °/s |
| angle at impact | **linear interpolation** of unwrapped roll onto `t_impact` | `t_impact` never lands on a sample (mean miss 4.4 ms, max 9.3 ms). A local *fit* biases the angle by up to 3.3° because the roll curve is strongly curved here; interpolation is unbiased and defined exactly at `t_impact` |
| roll rate at impact | **least-squares slope** of unwrapped roll over `|t − t_impact| ≤ SMOOTH_HALFWIDTH_S` | single-frame differencing has a worst-case one-sample sensitivity (1205 °/s) larger than the median value it reports |
| total angular speed | `‖rotvec(R₂ R₁⁻¹)‖ / (t₂ − t₁)` over the same time window | frame-independent; the roll plane carries a median of only 29 % of the true motion |
| window | time-based (`|t − t_impact| ≤ W`), **never** index-based `i ± k` | one file has a dropped 50 Hz deadline (a single 40 ms gap); an index window silently becomes asymmetric in time there |

**Three corrections to the briefing, all verified against the raw CSVs:**

1. "Each file contains exactly one duplicate frame" is wrong: there are **24 stale frames
   across the 18 files** (0 to 4 per file; 4 files have none). The de-duplication step is
   count-agnostic, so this only strengthens the case against single-frame differencing.
2. "Pitch stays within ±33°" is wrong: the full-flight euler-y range is **[−55.4°, +61.2°]**.
   Still far from gimbal lock, so the `xyz` decomposition is safe — but it is precisely why
   roll ≠ tilt (below).
3. One derivation reported the de-duplicated angle for `45deg_NT_2` as +54.35°; that value
   comes from deleting the *first* row of the identical pair and keeping the stale one's
   timestamp, which contradicts its own prose. Deleting the *later* (stale) row, as its prose
   says and as implemented here, gives **+57.43°**.

**Where the derivations disagreed, and what won:**

- **Rate estimator: ±60 ms straight-line fit vs. a ±2-sample windowed rotation.** They agree
  to <5 % on the roll channel, so this is a window-width question, not an estimator question.
  Settled at `SMOOTH_HALFWIDTH_S = 0.05 s`, which is the *time-based* window that reproduces
  the ±2-sample span (0.080 s median here) the only derivation that actually measured
  estimator sensitivity identified as the bias/variance knee: 7.1x smaller worst-case
  one-sample jump at 1 % amplitude cost, where widening to ~0.12 s throws away 23 % of the
  real signal. It also sits inside the band the timing derivation measured as immaterial
  (±40 ms vs ±60 ms moves the rate by a median 13 °/s).
- **Figures 2–3 metric: `|signed roll|` vs. `tilt`.** **Tilt wins**, and the evidence is not
  cosmetic. For an `xyz` euler decomposition `R[2,2] = cos(pitch)·cos(roll)` exactly, so
  `|roll|` is a *strict lower bound* on tilt and under-reports whenever pitch is non-zero —
  by up to **32.7°** here. At h = 2 m that flips the project's own success verdict on **3 of
  18 drops** (`180deg_NT_1`, `180deg_NT_2`, `180deg_WT_1`: all near-level in the roll plane
  while pitched 38–46° nose-down, which `|roll|` scores as a 9–16° landing and tilt correctly
  scores as a 39–48° one). Tilt is also frame-independent and is exactly what
  `docs/evaluate.py::tilt_deg` uses, so the 30° success line on figure 2 means what the
  project says it means.
- **Figure 3 magnitude: `|d roll/dt|` vs. 3-D `‖ω‖`.** **`‖ω‖` wins** — the two disagree by
  more than 25 % on 17 of 18 drops. `|d roll/dt|` would score `90deg_WT_3` as an essentially
  perfect landing (3.5 °/s) while the robot is in fact tumbling at 125 °/s.

**Mixed convention, stated explicitly:** figure 1 is a phase portrait, so its axes must be a
quantity and *its own* time derivative — signed roll and signed roll rate. Figures 2 and 3 are
magnitude comparisons, so they use the frame-independent pair (tilt, `‖ω‖`). **Figure 3 is
therefore not the absolute value of figure 1's y-axis**; the two differ by ~3.4x in typical
magnitude because most of the motion is out of the roll plane.

In [ ]:
# =============================================================================
# CONFIG -- this is the cell you edit.
# =============================================================================
import textwrap
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.spatial.transform import Rotation as Rot

try:                                    # notebook kernel
    from IPython.display import display
except ImportError:                     # plain `python3` execution
    display = print

# --- THE KNOB ----------------------------------------------------------------
DROP_HEIGHT_M = 2.0        # <<< drop height in metres. Re-run all three plots after changing.
# -----------------------------------------------------------------------------

G_M_S2 = 9.81              # gravity; free fall, no drag (drag over 0.64 s is far below the
                           # 20 ms drop-detection latency, so it is not worth modelling)
SMOOTH_HALFWIDTH_S = 0.05  # half-width of the time window used for BOTH rate estimates.
                           # Time-based, never index-based: one log has a 40 ms deadline miss.
UPRIGHT_DEG = 30.0         # success criterion, from docs/evaluate.py::UPRIGHT_DEG
MAX_LOGGED_HEIGHT_M = 4.81 # g * max(t_end)^2 / 2 -- the LONGEST log (0.9909 s) covers 4.816 m,
                           # so above this not one drop can be evaluated and the assert below
                           # stops the notebook. Every drop is evaluable up to 4.744 m (the
                           # SHORTEST log, 0.9835 s); in between, drops fall out one at a time and
                           # the loader excludes them individually rather than extrapolating.

# Series colors. Validated as a pair (CVD dE 8.0 protan, normal-vision dE 27.1, both >= 3:1
# contrast on a light surface). Do not substitute -- pure green #008300, for instance, fails
# against this orange at dE 3.2. The pair sits ON the CVD target boundary, so color is never
# the only channel: figure 1 uses two marker shapes and figures 2-3 hatch the WT bars.
COLOR_NT = "#eb6834"       # no tail   (orange)
COLOR_WT = "#1a9c6f"       # with tail (green)
VARIANT_COLOR = {"NT": COLOR_NT, "WT": COLOR_WT}
VARIANT_LABEL = {"NT": "No tail", "WT": "With tail"}
VARIANT_HATCH = {"NT": "", "WT": "///"}      # secondary encoding for figures 2-3
VARIANT_MARKER = {"NT": "o", "WT": "^"}      # secondary encoding for figure 1
RELEASE_ANGLES = (45, 90, 180)

# Ink + furniture. Text NEVER wears a series color.
INK, INK_MUTED, GRID_COLOR, SURFACE = "#1f2124", "#6b7076", "#dcdee1", "#ffffff"

assert DROP_HEIGHT_M > 0, "DROP_HEIGHT_M must be positive"
assert DROP_HEIGHT_M <= MAX_LOGGED_HEIGHT_M, (
    f"DROP_HEIGHT_M = {DROP_HEIGHT_M} m needs t_impact = {np.sqrt(2 * DROP_HEIGHT_M / G_M_S2):.4f} s "
    f"of flight, but the longest of the 18 logs ends at 0.9909 s ({MAX_LOGGED_HEIGHT_M} m). Every "
    "drop would have to be extrapolated, so there is nothing to plot."
)
if DROP_HEIGHT_M < 0.75:
    warnings.warn(
        f"h = {DROP_HEIGHT_M} m puts impact at t = {np.sqrt(2 * DROP_HEIGHT_M / G_M_S2):.3f} s, "
        "while the robot is still mid-flip. Drops can sit near +-180 deg there, where the SIGN "
        "of figure 1's x-axis is arbitrary. The (0,0)-centred framing is only honest above "
        "~0.75 m.", stacklevel=2)

T_IMPACT_S = float(np.sqrt(2.0 * DROP_HEIGHT_M / G_M_S2))
print(f"h = {DROP_HEIGHT_M} m  ->  t_impact = {T_IMPACT_S:.4f} s   "
      f"(rate window +-{SMOOTH_HALFWIDTH_S * 1e3:.0f} ms)")

# =============================================================================
# Shared figure furniture, so the three plot cells stay short and independent.
# =============================================================================
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.size": 10, "text.color": INK,
    "axes.labelcolor": INK, "axes.edgecolor": GRID_COLOR,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "xtick.labelcolor": INK, "ytick.labelcolor": INK,
    "hatch.linewidth": 1.3, "axes.axisbelow": True,
    "savefig.facecolor": SURFACE,
})


def style_axes(ax, *, grid_axis="both"):
    """Recessive grid behind the data, no top/right spines."""
    ax.grid(True, axis=grid_axis, color=GRID_COLOR, linewidth=0.7, zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID_COLOR)
    ax.tick_params(length=3, width=0.7)
    return ax


def caption(fig, text):
    """Wrap a caption to the figure width and reserve room for it -- never let it clip."""
    fig_w, fig_h = fig.get_size_inches()
    wrapped = textwrap.fill(text, width=int(fig_w * 72 * 0.92 / 4.4))
    n_lines = wrapped.count("\n") + 1
    fig.text(0.012, 0.012, wrapped, fontsize=7.6, color=INK_MUTED, ha="left", va="bottom")
    fig.tight_layout(rect=(0, 0.02 + n_lines * 1.35 * 7.6 / (fig_h * 72), 1, 1))


def grouped_bars(ax, table, value_col, ylabel, title):
    """Figures 2 and 3 share this: mean of n=3 with +-1 sd and the 3 trial dots on top."""
    width, offset = 0.18, 0.105     # two 0.18 bars + a 0.03 gap = 0.39 of a 1.0 slot; rest is air
    xs = np.arange(len(RELEASE_ANGLES), dtype=float)
    label_top = [0.0]

    for variant in ("NT", "WT"):
        means, sds, tops, positions = [], [], [], []
        for k, release in enumerate(RELEASE_ANGLES):
            vals = table.loc[(table["variant"] == variant) & (table["release_deg"] == release),
                             value_col].dropna().to_numpy()
            x = xs[k] + (-offset if variant == "NT" else offset)
            positions.append(x)
            mean = np.mean(vals) if vals.size else np.nan
            sd = np.std(vals, ddof=1) if vals.size > 1 else 0.0
            means.append(mean)
            sds.append(sd)
            # value label clears the error bar AND the highest trial dot, so nothing collides
            tops.append(max(mean + sd, vals.max()) if vals.size else np.nan)
            if vals.size:
                # trial dots: white fill + ink ring, so they read on both series colors
                jitter = np.linspace(-0.055, 0.055, vals.size) if vals.size > 1 else [0.0]
                ax.scatter(x + np.asarray(jitter), vals, s=24, facecolor="white",
                           edgecolor=INK, linewidth=0.9, zorder=5)
            if vals.size < 3:
                ax.annotate(f"n={vals.size}", (x, 0), xytext=(0, 6), textcoords="offset points",
                            ha="center", fontsize=8, color=INK_MUTED, zorder=6)
        means, sds = np.asarray(means, float), np.asarray(sds, float)
        ax.bar(positions, means, width, yerr=sds, capsize=4,
               color=VARIANT_COLOR[variant], hatch=VARIANT_HATCH[variant],
               edgecolor="white", linewidth=0.0,
               error_kw=dict(ecolor=INK, elinewidth=1.1, capthick=1.1, zorder=4),
               label=VARIANT_LABEL[variant], zorder=3)
        for x, mean, top in zip(positions, means, tops):
            if np.isfinite(mean):
                label_top.append(top)
                ax.annotate(f"{mean:.0f}", (x, top), xytext=(0, 7), textcoords="offset points",
                            ha="center", fontsize=9, color=INK_MUTED, zorder=6)

    ax.set_xticks(xs)
    ax.set_xticklabels([f"{r}° roll" for r in RELEASE_ANGLES])
    ax.set_xlabel("Release attitude", color=INK)
    ax.set_ylabel(ylabel, color=INK)
    ax.set_title(title, color=INK, fontsize=12, loc="left", pad=12)
    style_axes(ax, grid_axis="y")
    ax.set_xlim(-0.5, len(RELEASE_ANGLES) - 0.5)
    return float(np.nanmax(label_top))

In [ ]:
# =============================================================================
# LOADER -> one tidy row per drop.
# =============================================================================
FRONT_Q = ["F_Q0", "F_Q1", "F_Q2", "F_Q3"]     # (w, x, y, z), scalar FIRST, already world-aligned


def find_data_dir():
    """Locate the folder holding NT_0810/ and WT_0810/, relative to this notebook."""
    seeds = []
    try:                                        # .py execution
        seeds.append(Path(__file__).resolve().parent)
    except NameError:                           # notebook kernel: cwd is the notebook's dir
        pass
    seeds.append(Path.cwd())
    for seed in seeds:
        for base in (seed, *seed.parents):
            if (base / "NT_0810").is_dir() and (base / "WT_0810").is_dir():
                return base
            if (base / "experiments" / "NT_0810").is_dir():
                return base / "experiments"
    raise FileNotFoundError("could not find NT_0810/ and WT_0810/ near this notebook")


def wrap180(deg):
    """Fold degrees into (-180, 180] -- for DISPLAY only, never before differentiating."""
    return (np.asarray(deg) + 180.0) % 360.0 - 180.0


def drop_metrics(csv_path, variant, release_deg, trial):
    raw = pd.read_csv(csv_path)
    t_all = raw["Time"].to_numpy(float)
    t_all -= t_all[0]                              # Time is seconds since program start
    q_all = raw[FRONT_Q].to_numpy(float)

    # 1. De-duplicate stale serial reads (24 of them across the 18 files, 0-4 per file).
    #    A stale row repeats the previous measurement under a NEW timestamp, so the row to
    #    delete is the LATER one of the identical pair.
    keep = np.ones(len(q_all), bool)
    keep[1:] = np.any(q_all[1:] != q_all[:-1], axis=1)
    t, q = t_all[keep], q_all[keep]

    rot = Rot.from_quat(q, scalar_first=True)
    # 2. Unwrap BEFORE anything else: 4 files cross the +-180 branch cut.
    roll_unwrapped = np.degrees(np.unwrap(np.radians(rot.as_euler("xyz", degrees=True)[:, 0])))
    # tilt = angle between body +z and world +z, identical to docs/evaluate.py::tilt_deg
    tilt = np.degrees(np.arccos(np.clip(rot.as_matrix()[:, 2, 2], -1.0, 1.0)))

    covered = T_IMPACT_S <= t[-1]
    if covered:
        angle = float(wrap180(np.interp(T_IMPACT_S, t, roll_unwrapped)))
        tilt_at_impact = float(np.interp(T_IMPACT_S, t, tilt))
        window = np.abs(t - T_IMPACT_S) <= SMOOTH_HALFWIDTH_S
        idx = np.flatnonzero(window)
        if idx.size >= 2:
            # signed roll rate: least-squares slope, unbiased and stale-frame tolerant
            rate = float(np.polyfit(t[window] - T_IMPACT_S, roll_unwrapped[window], 1)[0])
            i1, i2 = idx[0], idx[-1]
            relative = rot[i2] * rot[i1].inv()     # 3-D windowed mean rotation
            omega = float(np.degrees(np.linalg.norm(relative.as_rotvec())) / (t[i2] - t[i1]))
            span = float(t[i2] - t[i1])
        else:
            rate = omega = span = np.nan
    else:
        angle = tilt_at_impact = rate = omega = span = np.nan

    return {
        "variant": variant, "variant_label": VARIANT_LABEL[variant],
        "release_deg": release_deg, "trial": trial, "file": csv_path.name,
        "roll_deg": angle, "roll_rate_dps": rate,
        "tilt_deg": tilt_at_impact, "omega_dps": omega,
        "upright": (tilt_at_impact < UPRIGHT_DEG) if covered else False,
        "release_roll_deg": float(rot.as_euler("xyz", degrees=True)[0, 0]),
        "n_rows": int(len(raw)), "n_stale_dropped": int((~keep).sum()),
        "t_end_s": float(t[-1]), "h_max_m": float(G_M_S2 * t[-1] ** 2 / 2),
        "window_span_s": span, "in_coverage": bool(covered),
    }


DATA_DIR = find_data_dir()
records = [drop_metrics(DATA_DIR / f"{v}_0810" / f"{r}deg_{v}success_0810_{n}.csv", v, r, n)
           for v in ("NT", "WT") for r in RELEASE_ANGLES for n in (1, 2, 3)]
drops = pd.DataFrame.from_records(records)

# --- refuse to extrapolate: a drop whose log ends before t_impact is dropped, loudly. ------
N_DROPS = len(drops)
missing = drops.loc[~drops["in_coverage"]]
N_MISSING = len(missing)
if N_MISSING:
    warnings.warn(f"{N_MISSING}/{N_DROPS} drops have no data at t_impact = {T_IMPACT_S:.4f} s "
                  f"and are excluded from all three figures.", stacklevel=2)
    print("=" * 78)
    print(f"!! h = {DROP_HEIGHT_M} m EXCEEDS LOG COVERAGE for {N_MISSING}/{N_DROPS} drops -- "
          "excluded, NOT extrapolated:")
    for _, row in missing.iterrows():
        print(f"   {row['file']:<34} t_end = {row['t_end_s']:.4f} s  <  "
              f"t_impact = {T_IMPACT_S:.4f} s   (max evaluable h = {row['h_max_m']:.3f} m)")
    print("=" * 78)

print(f"data: {DATA_DIR}")
print(f"{N_DROPS} drops, {N_DROPS - N_MISSING} evaluable at h = {DROP_HEIGHT_M} m; "
      f"{int(drops['n_stale_dropped'].sum())} stale frames removed; "
      f"rate window spans {drops['window_span_s'].min():.4f}-{drops['window_span_s'].max():.4f} s "
      f"({int(drops['upright'].sum())}/{N_DROPS} land under the {UPRIGHT_DEG:.0f} deg criterion)")

# The full per-drop table -- also the accessible, non-color view of every figure below.
display(drops[["variant_label", "release_deg", "trial", "file", "roll_deg", "roll_rate_dps",
               "tilt_deg", "omega_dps", "upright", "n_stale_dropped", "in_coverage"]]
        .round({"roll_deg": 2, "roll_rate_dps": 1, "tilt_deg": 2, "omega_dps": 1}))

## Figure 1 — where each drop is, and where it is going

One dot per drop, 18 in all. **x = signed roll at impact** (0° = level in the roll plane),
**y = signed roll rate at impact** (its time derivative). The origin is the ideal landing:
level *and* not rotating out of level. Distance from the centre is badness, so **read the
dots by how far out they sit, and in which quadrant**.

- Shape and color both carry morphology (circle/orange = no tail, triangle/green = with tail).
- Marker **size** carries the release angle — small 45°, medium 90°, large 180°.
- The shaded vertical band is `|roll| ≤ 30°`. Because `|roll| ≤ tilt` always, that band is a
  **necessary but not sufficient** condition for the 30° upright criterion — figure 2 has the
  real verdict.
- **The y-sign is phase, not quality.** The 45° drops oscillate through level (`45deg_NT_3`
  goes +44° → −25° → +28° within 0.64 s), so a positive rate there means "swinging back",
  not "diverging". Only for the 180° releases does a negative rate cleanly mean "still
  correcting toward level".
- Figure 3's angular speed is **not** the absolute value of this y-axis; it is the full 3-D
  spin, typically ~3.4x larger, because most of the motion is out of the roll plane.

In [ ]:
# =============================================================================
# FIGURE 1 -- square phase portrait, (0, 0) exactly centred.
# =============================================================================
pts = drops.dropna(subset=["roll_deg", "roll_rate_dps"])

# Symmetric limits ROUNDED OUT from the data, so the origin is dead centre.
lim_x = max(40.0, np.ceil(np.abs(pts["roll_deg"]).max() / 10.0) * 10.0)
lim_y = max(100.0, np.ceil(np.abs(pts["roll_rate_dps"]).max() / 50.0) * 50.0)
SIZE_FOR_RELEASE = {45: 90.0, 90: 165.0, 180: 260.0}    # >= 8 px across at every step

fig1, ax = plt.subplots(figsize=(7.2, 7.2))

# success band + zero crosshair, both behind the data
ax.axvspan(-UPRIGHT_DEG, UPRIGHT_DEG, color="#f0f1f2", zorder=1)
for edge in (-UPRIGHT_DEG, UPRIGHT_DEG):
    ax.axvline(edge, color=INK_MUTED, linestyle=(0, (4, 3)), linewidth=1.0, zorder=2)
ax.axhline(0, color="#b9bdc2", linewidth=1.0, zorder=2)
ax.axvline(0, color="#b9bdc2", linewidth=1.0, zorder=2)

for variant in ("NT", "WT"):
    sub = pts[pts["variant"] == variant]
    ax.scatter(sub["roll_deg"], sub["roll_rate_dps"],
               s=sub["release_deg"].map(SIZE_FOR_RELEASE), marker=VARIANT_MARKER[variant],
               facecolor=VARIANT_COLOR[variant], edgecolor="white", linewidth=1.6,
               zorder=4, label=VARIANT_LABEL[variant])

ax.set_xlim(-lim_x, lim_x)
ax.set_ylim(-lim_y, lim_y)
ax.set_box_aspect(1)                       # strictly square data box
ax.set_xlabel("Signed roll at impact (degrees)", color=INK)
ax.set_ylabel("Signed roll rate at impact (deg/s)", color=INK)
ax.set_title(f"Attitude vs. roll rate at impact  ·  h = {DROP_HEIGHT_M:g} m "
             f"(t = {T_IMPACT_S:.3f} s)  ·  {len(pts)} drops"
             + (f"\n{N_MISSING}/{N_DROPS} drops excluded: log ends before impact" if N_MISSING else ""),
             color=INK, fontsize=12, loc="left", pad=12)
style_axes(ax)
# band label rides the empty bottom edge, clear of every drop
ax.annotate(f"|roll| ≤ {UPRIGHT_DEG:.0f}°", xy=(0, -lim_y), xytext=(0, 9),
            textcoords="offset points", ha="center", va="bottom",
            fontsize=8.5, color=INK_MUTED)

shape_legend = ax.legend(loc="upper left", frameon=False, fontsize=9.5,
                         labelcolor=INK, handletextpad=0.6, borderpad=0.2)
ax.add_artist(shape_legend)
ax.legend(handles=[Line2D([], [], linestyle="none", marker="o", markerfacecolor="#c8ccd0",
                          markeredgecolor="white", markeredgewidth=1.2,
                          markersize=np.sqrt(SIZE_FOR_RELEASE[r]), label=f"{r}° release")
                   for r in RELEASE_ANGLES],
          loc="upper right", frameon=False, fontsize=9.5, labelcolor=INK,
          labelspacing=1.0, borderpad=0.2, handletextpad=0.8)

caption(fig1,
        "The origin is the ideal landing: level and not rotating out of level, so distance from "
        "the centre is badness. Front body only — the back IMU is dead in all 18 logs. "
        "t0 is drop DETECTION, so the "
        "release instant carries up to 20 ms of one-sided latency: worth ~2° typically and ~7° "
        "at worst on x, and up to 277 deg/s on y. The 45° drops oscillate through level, so the "
        "sign of y is the phase of that oscillation, not good vs. bad. The shaded band is a "
        "necessary but not sufficient condition for the 30° tilt criterion of figure 2.")
plt.show()

## Figure 2 — how level the robot is at impact

Body **tilt** at impact: the angle between the front body's +z and world +z, which is what
`docs/evaluate.py` scores. Lower is better; the dashed line is the project's own 30° upright
criterion. Bars are the **mean of 3 trials**, error bars are **±1 standard deviation**, and
the three white dots on each bar are the individual trials.

Read the dots first. The standard deviation is comparable to the mean in most cells, so a bar
that is lower than its neighbour is a hint, not a result — across all 9-vs-9 drops the
tail-vs-no-tail difference is not significant (Mann-Whitney p ≈ 0.22). The two cells that do
look real are the ends: **45° with-tail is a clean sweep** and **180° no-tail is a clean
failure**.

In [ ]:
# =============================================================================
# FIGURE 2 -- grouped bars, body tilt at impact.
# =============================================================================
fig2, ax = plt.subplots(figsize=(7.6, 5.2))

top = grouped_bars(ax, drops, "tilt_deg",
                   "Body tilt at impact (degrees)",
                   f"Tilt at impact  ·  h = {DROP_HEIGHT_M:g} m (t = {T_IMPACT_S:.3f} s)"
                   + (f"   — {N_MISSING}/{N_DROPS} drops excluded: log ends before impact"
                      if N_MISSING else ""))

ax.axhline(UPRIGHT_DEG, color=INK_MUTED, linestyle=(0, (5, 3)), linewidth=1.2, zorder=2)
# sits in the open air between two groups, so it never crosses a bar
ax.annotate(f"{UPRIGHT_DEG:.0f}° upright criterion (evaluate.py)",
            xy=(0.5, UPRIGHT_DEG), xytext=(0, 5), textcoords="offset points",
            ha="center", va="bottom", fontsize=8.5, color=INK_MUTED, zorder=7)
ax.set_ylim(0, max(top, UPRIGHT_DEG) * 1.30)
ax.legend(loc="upper right", frameon=False, fontsize=9.5, labelcolor=INK,
          handlelength=1.7, handleheight=1.3)

caption(fig2,
        "Bars are means of n = 3 trials; error bars are ± 1 sd; white dots are the individual "
        "trials — the sd is comparable to the mean in most cells, so read the dots. Tilt is "
        "frame-independent and is a STRICT UPPER BOUND on |roll|: it exceeds |roll| by up to 33° "
        "here (the 180° drops land nose-down), which flips the pass/fail verdict on 3 of the 18 "
        "drops. Front body only.")
plt.show()

## Figure 3 — how fast it is still spinning at impact

Same grouping, same bars-are-means-of-3 convention, but y is the **total 3-D angular speed**
`‖ω‖` at impact — how fast the front body is rotating about *any* axis, not just roll. Lower
is better; there is no published threshold, so no reference line.

This is deliberately not the absolute value of figure 1's y-axis. The roll plane carries a
median of only 29 % of the motion, so `|d roll/dt|` and `‖ω‖` disagree by more than 25 % on 17
of 18 drops — `90deg_WT_3` reads 3.5 °/s in roll while genuinely tumbling at 125 °/s.

In [ ]:
# =============================================================================
# FIGURE 3 -- grouped bars, total angular speed at impact.
# =============================================================================
fig3, ax = plt.subplots(figsize=(7.6, 5.2))

top = grouped_bars(ax, drops, "omega_dps",
                   "Total angular speed |ω| at impact (deg/s)",
                   f"Angular speed at impact  ·  h = {DROP_HEIGHT_M:g} m "
                   f"(t = {T_IMPACT_S:.3f} s)"
                   + (f"   — {N_MISSING}/{N_DROPS} drops excluded: log ends before impact"
                      if N_MISSING else ""))

ax.set_ylim(0, max(top, 1.0) * 1.30)
ax.legend(loc="upper right", frameon=False, fontsize=9.5, labelcolor=INK,
          handlelength=1.7, handleheight=1.3)

caption(fig3,
        "Bars are means of n = 3 trials; error bars are ± 1 sd; white dots are the individual "
        "trials. |ω| is the full 3-D spin rate, NOT the magnitude of figure 1's roll rate — it is "
        "~3.4x larger, because the roll plane carries a median of only 29 % of the motion. "
        "Estimated over a ± 50 ms window, within which ± 20 ms of release-timing latency is worth "
        "up to 277 deg/s. Front body only.")
plt.show()